In [0]:
src = spark.catalog.listTables('samples.accuweather')
for tbl in src:
  table_names = [tbl.name for tbl in src]
print(table_names)


In [0]:
for table_name in table_names:    
    
    create_query = f'''CREATE TABLE IF NOT EXISTS dev_bronze.researcher_api.{table_name} AS
    SELECT * FROM samples.accuweather.{table_name}'''

    spark.sql(create_query)

In [0]:
sql_query = f'''
select * from samples.accuweather.forecast_daily_calendar_imperial
-- ETL: Convert imperial values to metric units
create or replace table dev_silver.accuweather.forecast_daily_calendar_metric as
select
  city_name,
  country_code,
  latitude,
  longitude,
  date,
  -- Temperature: F to C
  (temperature_avg - 32) * 5/9 as temperature_avg_c,
  (temperature_max - 32) * 5/9 as temperature_max_c,
  (temperature_min - 32) * 5/9 as temperature_min_c,
  -- Precipitation: inches to mm
  precipitation_lwe_total * 25.4 as precipitation_lwe_total_mm,
  precipitation_lwe_rate_avg * 25.4 as precipitation_lwe_rate_avg_mm,
  precipitation_lwe_rate_max * 25.4 as precipitation_lwe_rate_max_mm,
  precipitation_lwe_rate_min * 25.4 as precipitation_lwe_rate_min_mm,
  -- Snow: inches to cm
  snow_total * 2.54 as snow_total_cm,
  snow_avg * 2.54 as snow_avg_cm,
  snow_max * 2.54 as snow_max_cm,
  snow_min * 2.54 as snow_min_cm,
  -- Wind speed: mph to km/h
  wind_speed_avg * 1.60934 as wind_speed_avg_kmh,
  wind_speed_max * 1.60934 as wind_speed_max_kmh,
  wind_speed_min * 1.60934 as wind_speed_min_kmh,
  wind_gust_avg * 1.60934 as wind_gust_avg_kmh,
  wind_gust_max * 1.60934 as wind_gust_max_kmh,
  wind_gust_min * 1.60934 as wind_gust_min_kmh,
  -- Visibility: miles to km
  visibility_avg * 1.60934 as visibility_avg_km,
  visibility_max * 1.60934 as visibility_max_km,
  visibility_min * 1.60934 as visibility_min_km,
  -- All other columns unchanged
  *
from samples.accuweather.forecast_daily_calendar_imperial''''